# Forward-Risk-Manager: Full Colab Pipeline

This notebook is set up to run the full workflow end-to-end:
1. Environment setup
2. Graph build
3. FF training
4. FF vs Backprop benchmark
5. FF sweep
6. Goodness backtest
7. Scenario book
8. Hallucination diagnostics


## 1) Enable GPU Runtime
Runtime -> Change runtime type -> Hardware accelerator: GPU

In [8]:
import torch
print('cuda:', torch.cuda.is_available())
print('torch:', torch.__version__)
print('cuda version:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Enable GPU runtime and rerun.')
print('gpu:', torch.cuda.get_device_name(0))


cuda: True
torch: 2.9.0+cu126
cuda version: 12.6
gpu: Tesla T4


## 2) Mount Drive and Open Repo

In [9]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/Forward-Risk-Manager'
if not Path(REPO_DIR).exists():
    raise FileNotFoundError(f'Repo path not found: {REPO_DIR}')

%cd $REPO_DIR
print('cwd:', os.getcwd())
!ls -la


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Forward-Risk-Manager
cwd: /content/drive/MyDrive/Forward-Risk-Manager
total 54
drwx------ 2 root root  4096 Feb  9 02:45 configs
drwx------ 4 root root  4096 Feb  8 00:34 data
drwx------ 2 root root  4096 Feb  9 16:29 .git
-rw------- 1 root root   127 Feb  8 00:13 .gitignore
drwx------ 2 root root  4096 Feb  9 03:04 notebooks
-rw------- 1 root root   386 Feb  9 02:49 pyproject.toml
-rw------- 1 root root 12669 Feb  9 18:08 README.md
drwx------ 2 root root  4096 Feb  9 18:07 reports
-rw------- 1 root root    95 Feb  5 23:01 requirements.txt
drwx------ 2 root root  4096 Feb  9 18:06 runs
drwx------ 3 root root  4096 Feb  9 02:45 scripts
drwx------ 4 root root  4096 Feb  9 17:18 src
drwx------ 3 root root  4096 Feb  9 18:16 tests
drwx------ 2 root root  4096 Feb  5 23:02 .venv


## 3) Install Dependencies

In [10]:
!pip install -r requirements.txt
!pip install -e .


Obtaining file:///content/drive/MyDrive/Forward-Risk-Manager
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for forward-risk-manager (pyproject.toml) ... done
  Created wheel for forward-risk-manager: filename=forward_risk_manager-0.1.0-0.editable-py3-none-any.whl size=6047 sha256=122e42f8a7b95a10c1a2262d3c5f44d71056430d4d2a2142ad500bc2714e08de
  Stored in directory: /tmp/pip-ephem-wheel-cache-qn_0jp24/wheels/01/78/64/2a7fcf2ce178fa37b035fda1b3c1b372ca2d55b4bf5e858d9f
Successfully built forward-risk-manager
  Attempting uninstall: forward-risk-manager
    Found existing installation: forward-risk-manager 0.1.0
    Uninstalling forward-risk-manager-0.1.0:
      Successfully uninstalled forward-risk-manager-0.1.0


## 4) Pipeline Configuration
Set these once, then run the full pipeline cell below.

In [11]:
from pathlib import Path
import tomllib

# Build config should match the train graph path you want to use
BUILD_CONFIG = 'configs/long_constituents.toml'
TRAIN_CONFIG = 'configs/train_long_constituents.toml'

DEVICE = 'cuda'

RUN_BUILD = True
RUN_TRAIN = True
RUN_BENCHMARK = True
RUN_SWEEP = True
RUN_PLOT_SWEEP = True
RUN_PROMOTE_SWEEP = True
RUN_RETRAIN_AFTER_PROMOTE = False
RUN_REBENCHMARK_AFTER_PROMOTE = False
RUN_GOODNESS_BACKTEST = True
RUN_SCENARIO_BOOK = True
RUN_HALLUCINATION_DIAGNOSTICS = True

ALLOW_OPTIONAL_FAILURES = True
SCENARIO_FALLBACK_TICKER = 'MDY'

SWEEP_CSV = 'reports/ff_sweep.csv'
PROMOTE_SWEEP_RANK_BY = 'auto'  # auto | score | eval_sep | eval_acc | graphs_per_s
PROMOTE_SWEEP_MODE = ''          # e.g. 'ff_e2e' to force a mode, else '' for any
PROMOTE_SWEEP_APPLY_MODE = True

BACKTEST_TICKER = 'BWC'
SCENARIO_TICKER = 'BWC'
SCENARIO_NUM = 10
SCENARIO_TARGET_DROP = -0.10
SCENARIO_CONSTRAINT_WEIGHT = 10.0

for p in [BUILD_CONFIG, TRAIN_CONFIG]:
    if not Path(p).exists():
        raise FileNotFoundError(p)

with Path(TRAIN_CONFIG).open('rb') as f:
    train_cfg = tomllib.load(f)
print('train graphs path:', train_cfg.get('train', {}).get('graphs', 'n/a'))
print('build config:', BUILD_CONFIG)
print('train config:', TRAIN_CONFIG)


train graphs path: data/processed_long/graphs_constituents.pt
build config: configs/long_constituents.toml
train config: configs/train_long_constituents.toml


## 5) Helpers

In [12]:
import shlex
import subprocess
import sys
import time

PYTHON = shlex.quote(sys.executable)

def run(cmd: str, allow_fail: bool = False) -> bool:
    print('\n' + '=' * 100)
    print(cmd)
    print('=' * 100)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True)
    dt = time.time() - t0
    if p.returncode != 0:
        msg = f'Command failed ({p.returncode}): {cmd}'
        if allow_fail:
            print(f'WARNING: {msg}')
            return False
        raise RuntimeError(msg)
    print(f'Completed in {dt / 60:.2f} min')
    return True


## 6) Run Full Pipeline

In [13]:
if RUN_BUILD:
    run(f"{PYTHON} scripts/build_graphs.py --config {shlex.quote(BUILD_CONFIG)}")

if RUN_TRAIN:
    run(
        f"{PYTHON} scripts/train_ff_gnn.py --config {shlex.quote(TRAIN_CONFIG)} --device {shlex.quote(DEVICE)}"
    )

if RUN_BENCHMARK:
    run(f"{PYTHON} scripts/benchmark_training.py --config {shlex.quote(TRAIN_CONFIG)}")

if RUN_SWEEP:
    run(f"{PYTHON} scripts/ff_sweep.py --config {shlex.quote(TRAIN_CONFIG)}")

if RUN_PLOT_SWEEP:
    run(f"{PYTHON} scripts/plot_ff_sweep.py --csv {shlex.quote(SWEEP_CSV)}")

if RUN_PROMOTE_SWEEP:
    promote_mode_arg = f" --mode {shlex.quote(PROMOTE_SWEEP_MODE)}" if PROMOTE_SWEEP_MODE else ''
    promote_apply_mode_arg = '--apply-mode' if PROMOTE_SWEEP_APPLY_MODE else '--no-apply-mode'
    run(
        f"{PYTHON} scripts/promote_sweep_best.py --config {shlex.quote(TRAIN_CONFIG)} "
        f"--csv {shlex.quote(SWEEP_CSV)} "
        f"--rank-by {shlex.quote(PROMOTE_SWEEP_RANK_BY)}"
        f"{promote_mode_arg} --apply {promote_apply_mode_arg}"
    )

    with Path(TRAIN_CONFIG).open('rb') as f:
        train_cfg = tomllib.load(f)
    print('updated train goodness_target:', train_cfg.get('train', {}).get('goodness_target', 'n/a'))

    if RUN_RETRAIN_AFTER_PROMOTE:
        run(
            f"{PYTHON} scripts/train_ff_gnn.py --config {shlex.quote(TRAIN_CONFIG)} --device {shlex.quote(DEVICE)}"
        )
        if RUN_REBENCHMARK_AFTER_PROMOTE:
            run(f"{PYTHON} scripts/benchmark_training.py --config {shlex.quote(TRAIN_CONFIG)}")

if RUN_GOODNESS_BACKTEST:
    run(
        f"{PYTHON} scripts/goodness_backtest.py --config {shlex.quote(TRAIN_CONFIG)} "
        f"--ticker {shlex.quote(BACKTEST_TICKER)} --horizons 5,21"
    )

if RUN_SCENARIO_BOOK:
    scenario_ticker = (SCENARIO_TICKER or '').strip().upper()
    try:
        import torch

        graphs_path = Path(train_cfg.get('train', {}).get('graphs', ''))
        ticker_set = set()
        if graphs_path.exists():
            try:
                payload = torch.load(graphs_path, map_location='cpu', weights_only=False)
            except TypeError:
                payload = torch.load(graphs_path, map_location='cpu')
            if isinstance(payload, dict):
                tickers_list = payload.get('tickers', []) or []
                ticker_set = {t for tickers in tickers_list for t in tickers}

        if scenario_ticker and ticker_set and scenario_ticker not in ticker_set:
            fallback = (SCENARIO_FALLBACK_TICKER or '').strip().upper()
            if fallback and fallback in ticker_set:
                print(f"[scenario_book] {scenario_ticker} not found in graphs. Using fallback {fallback}.")
                scenario_ticker = fallback
            else:
                scenario_ticker = sorted(ticker_set)[0]
                print(f"[scenario_book] {SCENARIO_TICKER} not found in graphs. Using {scenario_ticker}.")
        elif scenario_ticker and not ticker_set:
            print('[scenario_book] Graph payload has no ticker metadata; running without target ticker constraint.')
            scenario_ticker = ''
    except Exception as e:
        print(f'[scenario_book] Ticker precheck skipped: {e}')

    ticker_arg = f" --target-ticker {shlex.quote(scenario_ticker)}" if scenario_ticker else ''
    run(
        f"{PYTHON} scripts/scenario_book.py --config {shlex.quote(TRAIN_CONFIG)} "
        f"--num-scenarios {SCENARIO_NUM}{ticker_arg} "
        f"--target-drop {SCENARIO_TARGET_DROP} "
        f"--constraint-weight {SCENARIO_CONSTRAINT_WEIGHT}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
    )

if RUN_HALLUCINATION_DIAGNOSTICS:
    run(
        f"{PYTHON} scripts/plot_hallucination.py --config {shlex.quote(TRAIN_CONFIG)} "
        f"--save-csv-all reports/hallucination_window_all.csv"
    )
    run(f"{PYTHON} scripts/hallucination_calibration.py --csv reports/hallucination_window_all.csv")
    run(f"{PYTHON} scripts/plot_hallucination_diagnostics.py --csv reports/hallucination_window_all.csv")
    run(f"{PYTHON} scripts/stress_test_report.py --csv reports/hallucination_window_all.csv")

print('Pipeline finished.')



/usr/bin/python3 scripts/build_graphs.py --config configs/long_constituents.toml
Completed in 2.10 min

/usr/bin/python3 scripts/train_ff_gnn.py --config configs/train_long_constituents.toml --device cuda
Completed in 10.25 min

/usr/bin/python3 scripts/benchmark_training.py --config configs/train_long_constituents.toml
Completed in 0.54 min

/usr/bin/python3 scripts/ff_sweep.py --config configs/train_long_constituents.toml
Completed in 0.31 min

/usr/bin/python3 scripts/plot_ff_sweep.py --csv reports/ff_sweep.csv
Completed in 0.06 min

/usr/bin/python3 scripts/goodness_backtest.py --config configs/train_long_constituents.toml --ticker BWC --horizons 5,21
Completed in 0.41 min

/usr/bin/python3 scripts/scenario_book.py --config configs/train_long_constituents.toml --num-scenarios 10 --target-ticker BWC --target-drop -0.1 --constraint-weight 10.0


RuntimeError: Command failed (1): /usr/bin/python3 scripts/scenario_book.py --config configs/train_long_constituents.toml --num-scenarios 10 --target-ticker BWC --target-drop -0.1 --constraint-weight 10.0

## 7) Inspect Key Outputs

In [ ]:
from pathlib import Path

report_files = sorted(Path('reports').glob('*'))
print(f'reports files: {len(report_files)}')
for p in report_files[:80]:
    print(p)

print('\nRun artifacts:')
for p in sorted(Path('runs').glob('*')):
    print(p)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

images = [
    'reports/ff_train_long_constituents.png',
    'reports/benchmark.png',
    'reports/benchmark_speed_sep.png',
    'reports/ff_sweep_tradeoff.png',
    'reports/ff_sweep_pareto.png',
    'reports/goodness_scatter.png',
    'reports/hallucination_plot.png',
]

for img_path in images:
    p = Path(img_path)
    if not p.exists():
        continue
    img = Image.open(p)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(str(p))
    plt.show()


## Notes

- If you hit GPU OOM, reduce `batch_size` in `TRAIN_CONFIG` and/or lower `hidden_dim`.
- `scenario_book` auto-falls back to a valid ticker if `SCENARIO_TICKER` is missing from graph metadata.
- `RUN_PROMOTE_SWEEP` applies best sweep params into `[train]` via `scripts/promote_sweep_best.py`.
- Set `RUN_RETRAIN_AFTER_PROMOTE = True` to immediately retrain with promoted params in the same notebook run.
- Set `ALLOW_OPTIONAL_FAILURES = False` if you want the notebook to stop immediately on optional stage failures.
- For full-universe runs, switch to `configs/long_alltickers.toml` + `configs/train_long_alltickers.toml`.
